# sosci-pipeline v2.2 — Colab runner
**Tick-Borne Diseases / Climate / Sub-Saharan Africa**

This run picks up from your CURRENT state: `master_canonical_screen_ready.parquet`
already exists. This notebook:
1. Re-enriches any stragglers left unmatched by API rate limits (optional, repeatable)
2. Runs `tick_relevance` against the screen-ready file (title+abstract+keywords are
   most complete at this point — confirmed best to run it here)
3. Lets you screen with any subset of the 3 models (each is independently optional)
4. Runs consensus + PRISMA, then the bibliometric analysis layer

**Reminder on the screening question** (config: `configs/ticks_climate_ssa.yaml`,
section `screening.question`): the prompt asks whether a study evaluates the impact
of **climate on ticks** — not just "is this about ticks". `tick_relevance` (step 2
above) is the separate, free, rule-based check for "is this about ticks at all",
run BEFORE the paid LLM climate-impact screening so the 3 models aren't spending
calls on records that are obviously off-topic.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install + load the package

In [ ]:
!pip install pyarrow pyyaml rapidfuzz openpyxl requests numpy scipy -q
!pip install anthropic openai google-genai scikit-learn -q
!pip install pycountry statsmodels pymannkendall bertopic sentence-transformers umap-learn hdbscan -q

import zipfile, sys, os
# ZIP = '/content/drive/MyDrive/sosci-pipeline.zip'   # <-- EDIT to your zip path
ZIP='/content/drive/MyDrive/COGNOSX/sosci-pipeline.zip'
zipfile.ZipFile(ZIP).extractall('/content/sosci_pkg')
sys.path.insert(0, '/content/sosci_pkg/sosci-pipeline')
print('loaded')

## 3. Build context (your existing v2 folder)

In [ ]:
from sosci_pipeline import runner, provenance
CONFIG = '/content/sosci_pkg/sosci-pipeline/configs/ticks_climate_ssa.yaml'   # <-- EDIT if needed
ctx = runner.build_ctx(CONFIG, work_subdir='preprocessing_data_v2')
print('files currently in work dir:')
print(sorted(f for f in os.listdir(ctx.work) if f.endswith('.parquet')))

## 4. API keys via Colab Secrets 🔑 (needed for screening, not for re-enrich/tick_relevance)

In [ ]:
from google.colab import userdata
for k in ['ANTHROPIC_API_KEY','OPENAI_API_KEY','GEMINI_API_KEY']:
    try: os.environ[k] = userdata.get(k); print(k, 'loaded')
    except Exception: print(k, 'NOT set')

## 5. Re-enrich stragglers (optional, repeatable)
Retries ONLY records still unmatched (`oa_match_method == ""`) — leaves everything
already matched untouched, and backs up the file before overwriting. Re-run this
cell as many times as you like:
- with the SAME email after waiting a while for rate limits to reset, or
- with a DIFFERENT email (`EMAIL_OVERRIDE`) right away for a fresh polite-pool quota

Safe to skip entirely if you're not worried about rate-limited stragglers.

In [ ]:
from sosci_pipeline.reenrich_stragglers import run_reenrich_stragglers

EMAIL_OVERRIDE = 'cognosxx@gmail.com' # None   # e.g. 'your-other-email@example.com'; None = use the config's email
MAX_RECORDS = None      # e.g. 5000 to test on a subset first; None = all stragglers

run_reenrich_stragglers(ctx, email_override=EMAIL_OVERRIDE, max_records=MAX_RECORDS)

In [ ]:
for st in ['fix_funders','dedupe_pass2','coverage','type_audit','reenrich','readiness','backfill']:
    runner.run_stage(st, ctx)

In [ ]:
from sosci_pipeline.reenrich_stragglers import run_reenrich_stragglers

EMAIL_OVERRIDE = 'cognosxx@gmail.com' # None   # e.g. 'your-other-email@example.com'; None = use the config's email
MAX_RECORDS = None      # e.g. 5000 to test on a subset first; None = all stragglers

run_reenrich_stragglers(ctx, email_override=EMAIL_OVERRIDE, max_records=MAX_RECORDS)

In [ ]:
for st in ['fix_funders','dedupe_pass2','coverage','type_audit','reenrich','readiness','backfill']:
    runner.run_stage(st, ctx)

## 6. Tick-relevance check (free, rule-based — run before screening)
Confirms each record is genuinely about ticks (title/abstract/keywords now most
complete, since this runs after enrichment). Flags rather than drops — irrelevant
records are saved separately for spot-checking, not deleted.

In [ ]:
runner.run_stage('tick_relevance', ctx)

In [ ]:
runner.run_stage('tick_relevance', ctx)

In [ ]:
# runner.run_stage('tick_relevance', ctx)

## 7. Screen — choose which model(s) to run
Each model is independent and optional. Toggle below, then run the cell.
Re-run any time to add a model you skipped earlier, or to resume an interrupted pass.

In [ ]:
from sosci_pipeline import screen


RUN_OPENAI    = False
RUN_ANTHROPIC = False
RUN_GOOGLE    = False

# RUN_OPENAI    = True
# RUN_ANTHROPIC = True
# RUN_GOOGLE    = True

if RUN_OPENAI:
    screen.run_screen(ctx, model_label='openai')

In [ ]:
if RUN_ANTHROPIC:
    screen.run_screen(ctx, model_label='anthropic')

In [ ]:
if RUN_GOOGLE:
    screen.run_screen(ctx, model_label='google')

## 8. Consensus + PRISMA
Works with 1, 2, or 3 models present — re-run after adding more.

In [ ]:
# screen.run_consensus(ctx)
P, path = provenance.write_prisma(ctx.work)
import json; print(json.dumps({k: v for k, v in P.items() if v is not None}, indent=2))

## 9. Validation (optional, for publishable screening metrics)
Step A draws a stratified sample; hand-label `human_label`, then run Step B.

In [ ]:
# screen.draw_validation_sample(ctx)
# # ... open validation_sample.xlsx, fill human_label (include/exclude), save, then:
# # screen.compute_validation(ctx)

## 10. Bibliometric analysis layer (optional, after screening)
Topic modeling → the three analysis layers (research output vs. burden-proxy
equity, topic composition by region, climate-as-emerging-niche).

In [ ]:
runner.run_stage('topic_model', ctx)

### Layer 1 — research output vs. tick-borne disease relevance
For the BARI-style regression, download the Sambado & Ryan (2026) supplementary
trait matrix from Dryad (doi:10.5061/dryad.1g1jwsv9w) and save it as
`sambado_global_trait_matrix.csv` in the work folder first. The country output
table is produced either way.

In [ ]:
runner.run_stage('layer1_burden', ctx)

### Layer 2 — topic composition by region (SSA vs. global)

In [ ]:
runner.run_stage('layer2_topic_region', ctx)

### Layer 3 — climate-tick research as an emerging niche

In [ ]:
runner.run_stage('layer3_climate_niche', ctx)